In [ ]:
%load_ext autoreload
%autoreload 2

# modal

> Abstractions for creating a Modal Sandbox

In [ ]:
#| default_exp modal

In [ ]:
#| export
import modal

In [ ]:
#| export
def mk_app(name: str        # App name to look up or create
           ) -> modal.App:
    "Look up a Modal App by name, creating it if missing."
    return modal.App.lookup(name, create_if_missing=True)

In [ ]:
app = mk_app('solveit-sandbox'); app

In [ ]:
#| export
_default_condas = ['ipykernel', 'fastai', 'cuda-toolkit',]

In [ ]:
#| export
_default_pips = [
    'ipykernel',
    'fastai', 'transformers', 'diffusers', 'accelerate', 'datasets',
    'huggingface_hub', 'peft', 'safetensors', 'sentence-transformers',
    'xformers', 'bitsandbytes', 'ninja', 'einops',
    'wandb',
    'gradio', 'python-fasthtml', 'plotly', 'ipywidgets',
    'fsspec', 's3fs', 'gcsfs',
    'librosa', 'imageio', 'imageio-ffmpeg',
]

In [ ]:
#| export
def mk_img(
    pips  :list|None=None,  # pip packages (defaults to _default_pips)
) -> modal.Image:
    "Create a Modal Image with system + Python packages."
    if pips   is None: pips   = _default_pips
    return modal.Image.debian_slim().apt_install('openssh-server').pip_install(pips)

In [ ]:
img = mk_img(); img

Image(<function _Image.pip_install.<locals>.build_dockerfile at 0x7ac9fb20e0c0>)

In [ ]:
#| export
def get_apts() -> list[str]:
    "List installed system package names."
    return run('dpkg-query -W -f=\'${Package}\\n\'').splitlines()

In [ ]:
get_apts()[:10]

['adduser',
 'alsa-topology-conf',
 'alsa-ucm-conf',
 'apt',
 'aria2',
 'autoconf',
 'automake',
 'autotools-dev',
 'base-files',
 'base-passwd']

In [ ]:
#| export
def get_pips() -> list[str]:
    "List installed Python packages (name==version)."
    return [l for l in run('pip freeze').splitlines() if not l.startswith(('-e', '@'))]

In [ ]:
get_pips()[:10]

['accelerate==1.13.0',
 'anki==25.9.2',
 'beartype==0.22.5',
 'blis==1.3.0',
 'catalogue==2.0.10',
 'cbor2==5.8.0',
 'cloudpathlib==0.23.0',
 'cloudpickle==3.1.2',
 'confection==0.1.5',
 'coolname==2.2.0']

In [ ]:
#| export
import json

In [ ]:
#| export
def get_secrets() -> dict[str,str]:
    "Secrets from solveit_settings.json."
    with open('/app/data/solveit_settings.json') as f: return json.load(f)['secrets']

In [ ]:
#| export
def mk_sandbox(
    app    :modal.App,              # Modal App to register the Sandbox with
    img    :modal.Image,            # Image for the Sandbox environment
    timeout:int        =600,        # auto-terminate after this many seconds
    gpu    :str|None    =None,      # GPU type (e.g. "T4", "A100") or None
    secrets:dict|None   =None,      # secrets dict (defaults to local solveit)
) -> modal.Sandbox:
    "Create a Modal Sandbox with optional GPU and SSH."
    if secrets is None: secrets = get_secrets()
    print(f'⌛ creating sandbox…')
    sb = modal.Sandbox.create('sleep', 'infinity',
        app=app, image=img, timeout=timeout,
        unencrypted_ports=[22], gpu=gpu,
        secrets=[modal.Secret.from_dict(secrets)])
    print('✔ sandbox ready')
    return sb

In [ ]:
sb = mk_sandbox(app, img, gpu='T4'); sb

⌛ creating sandbox…


✔ sandbox ready


Sandbox()

In [ ]:
#| export
def get_tunnel(sb: modal.Sandbox  # Sandbox with an exposed TCP tunnel
               ) -> tuple[str,int]:
    "Get unencrypted host and port for a Sandbox's TCP tunnel."
    t = sb.tunnels()[22]
    host, port = t.unencrypted_host, t.unencrypted_port
    print(f'✔ tunnel: {host}:{port}')
    return host, port

In [ ]:
host,port = get_tunnel(sb); host,port

✔ tunnel: r442.modal.host:44061


('r442.modal.host', 44061)

In [ ]:
#| export
import os

In [ ]:
#| export
def get_pubkey() -> str:
    "SSH public key from environment."
    return os.environ['SSH_PUBKEY']

In [ ]:
#| export
def inject_pubkey(sb    :modal.Sandbox,  # Sandbox to inject the key into
                  pubkey:str             # SSH public key string
                  ):
    "Inject an SSH public key into the Sandbox's authorized_keys."
    sb.exec('mkdir', '-p', '/root/.ssh')
    sb.exec('bash', '-c', f'echo {pubkey} > /root/.ssh/authorized_keys')
    print('✔ public key injected')

In [ ]:
inject_pubkey(sb, get_pubkey())

✔ public key injected


In [ ]:
#| export
from time import sleep

In [ ]:
#| export
def start_ssh(sb: modal.Sandbox  # Sandbox with SSH server installed
              ):
    "Start SSH service in the Sandbox, waiting up to ~6 seconds for it to come online."
    sb.exec('service', 'ssh', 'start')
    for _ in range(20):
        proc = sb.exec('service', 'ssh', 'status')
        if 'fail' in proc.stdout.read(): sleep(0.3)
        elif 'run' in proc.stdout.read():
            print('✔ started ssh service')
            return
    print('✖ failed to start ssh service')

In [ ]:
start_ssh(sb)

✔ started ssh service


In [ ]:
#| export
from fastcore.all import *

In [ ]:
#| export
def mk_ssh(host: str,  # Tunnel host
           port: int,  # Tunnel port
           ) -> object:
    "Return an SSH function for the given Modal tunnel."
    return lambda *cmd: run('ssh', '-p', str(port),
        '-o', 'StrictHostKeyChecking=no',
        '-o', 'UserKnownHostsFile=~/.ssh/known_hosts',
        '-o', 'ConnectTimeout=10',
        f'root@{host}', *cmd)

In [ ]:
ssh = mk_ssh(host,port); ssh

<function __main__.mk_ssh.<locals>.<lambda>(*cmd)>

In [ ]:
#| export
def verify_ssh(ssh: object  # SSH function returned by mk_ssh
               ):
    "Verify SSH connection to Sandbox and print system info."
    h, u, w = ssh('hostname; uname -srmo; whoami').splitlines()[:3]
    sys_name, ver, arch, os_name = u.split()
    gpu = ssh('nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo "no GPU"').strip()
    print(f'System：  {sys_name}')
    print(f'Hostname：{h}')
    print(f'User：  {w}')
    print(f'Kernel：  {ver}')
    print(f'Architecture：  {arch}')
    print(f'OS Type：{os_name}')
    print(f'GPU：   {gpu}')

In [ ]:
verify_ssh(ssh)

系统：  Linux
主机名：modal
用户：  root
内核：  4.4.0
架构：  x86_64
系统类型：GNU/Linux


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()